In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python matplotlib

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

# 2. Baixar o checkpoint do modelo SAM usando urllib (sem precisar da biblioteca wget)
checkpoint_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
checkpoint_path = "sam_vit_h_4b8939.pth"

if not os.path.exists(checkpoint_path):
    print("Baixando o modelo SAM (vit_h)... isso pode levar um minuto.")
    urllib.request.urlretrieve(checkpoint_url, checkpoint_path)
    print("Download concluído!")
file_name = "revapFumaca.jpg"

# Tenta carregar com extensões comuns se não tiver especificado
import os
for ext in [".jpg", ".png", ".jpeg", ".JPG", ".PNG"]:
    if os.path.exists(file_name + ext):
        file_name = file_name + ext
        break

image = cv2.imread(file_name)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 2. Exibir a imagem com os eixos para identificar as coordenadas (x, y)
plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.title("Imagem com Eixos para Identificação de Coordenadas (x, y)")
plt.xlabel("Eixo X (Largura em pixels)")
plt.ylabel("Eixo Y (Altura em pixels)")
plt.grid(True, which='both', color='red', linestyle='--', linewidth=0.5)
plt.show()

print(f"Dimensões da imagem: {image.shape[1]}px de largura por {image.shape[0]}px de altura.")

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

# 1. Carregar o predictor do SAM com a GPU/CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_h"](checkpoint=checkpoint_path)
sam.to(device=device)
predictor = SamPredictor(sam)

# Definir a imagem no predictor
predictor.set_image(image)

def plot_segmentation_comparison(image, mask, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Coluna 1: Imagem Original
    axes[0].imshow(image)
    axes[0].set_title("Imagem Original", fontsize=12)
    axes[0].axis('off')

    # Coluna 2: Máscara Binária
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Máscara Binária (Fumaça)", fontsize=12)
    axes[1].axis('off')

    # Coluna 3: Máscara Sobreposta à Imagem
    overlay = image.copy()
    overlay[mask] = [255, 0, 0] # Destaca a máscara em vermelho
    blended = cv2.addWeighted(image, 0.65, overlay, 0.35, 0)

    axes[2].imshow(blended)
    axes[2].set_title("Sobreposição na Imagem", fontsize=12)
    axes[2].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# CONFIGURAÇÃO 1: Apenas Pontos Positivos no Corpo da Fumaça
input_points_1 = np.array([[400, 200], [450, 300]])
input_labels_1 = np.array([1, 1]) # 1 = Ponto Positivo

masks_1, _, _ = predictor.predict(
    point_coords=input_points_1,
    point_labels=input_labels_1,
    multimask_output=False
)
plot_segmentation_comparison(image, masks_1[0], "Configuração 1: Prompts de Pontos Positivos")

# CONFIGURAÇÃO 2: Caixa Delimitadora (Bounding Box)
input_box_2 = np.array([250, 50, 650, 420]) # [x_min, y_min, x_max, y_max]

masks_2, _, _ = predictor.predict(
    box=input_box_2,
    multimask_output=False
)
plot_segmentation_comparison(image, masks_2[0], "Configuração 2: Bounding Box Envolvendo a Pluma")

# CONFIGURAÇÃO 3: Pontos Positivos + Exclusão Negativa (Céu/Nuvens)

input_points_3 = np.array([
    [400, 200], # Positivo: Fumaça superior
    [450, 300], # Positivo: Fumaça média
    [850, 100], # Negativo: Céu no canto superior direito
    [150, 100]  # Negativo: Céu no canto superior esquerdo
])
input_labels_3 = np.array([1, 1, 0, 0]) # 1 = incluir, 0 = excluir

masks_3, _, _ = predictor.predict(
    point_coords=input_points_3,
    point_labels=input_labels_3,
    multimask_output=False
)
plot_segmentation_comparison(image, masks_3[0], "Configuração 3: Pontos Positivos + Exclusão Negativa do Céu")


Alunos: Sávio Mendes (RA: 163991) | Paulo Felipe Crot Ramos (RA: 169055)